In [ ]:
import re 
import pandas as pd
from collections import Counter


In [ ]:
df = pd.read_csv("/Users/kaitaoyang/Downloads/datasets_conversations/DailyDialog/train.csv")
print(df.shape)
df.head(2)

In [ ]:
text_raw = " ".join(df.dialog.to_list()).lower()
text_clean_list = re.findall(r"[a-zA-Z\-]+", text_raw)
text_clean_list[:100]

In [ ]:
len(set(text_clean_list))

In [ ]:
word_counter = Counter(text_clean_list)
word_counter.most_common(1000)


In [ ]:
# pip install google-cloud-translate
from google.oauth2 import service_account
from google.cloud import translate_v2 as translate

# Load credentials from the JSON file
credentials = service_account.Credentials.from_service_account_file("../kt-languages-148ff3e33c54.json")

# Initialize client
translate_client = translate.Client(credentials=credentials)

# Example usage
result = translate_client.translate("Hello world", target_language="ru")
print(result["translatedText"])




In [ ]:
import argostranslate.package
import argostranslate.translate

# Update model index (list of all available models)
argostranslate.package.update_package_index()
available_packages = argostranslate.package.get_available_packages()

# Find the English → Russian package
package_to_install = next(
    pkg for pkg in available_packages
    if pkg.from_code == "en" and pkg.to_code == "ru"
)

# Install the package
argostranslate.package.install_from_path(package_to_install.download())

# Now load installed languages
installed_languages = argostranslate.translate.get_installed_languages()
from_lang = next(lang for lang in installed_languages if lang.code == "en")
to_lang = next(lang for lang in installed_languages if lang.code == "ru")

translation = from_lang.get_translation(to_lang)

# Test translation
print(translation.translate("Hello world, how are you?"))


In [ ]:
# Apply row by row, keeping partial results
df["dialog_ru"] = df["dialog"].apply(translation.translate)

# Save partial progress, so you don’t lose already translated results
df.to_csv("translated_partial.csv", index=False, encoding="utf-8-sig")

In [ ]:
df.to_csv("translated_partial_ru.csv", index=False, encoding="utf-8-sig")

df["dialog_ru"].iloc[0]

In [ ]:
df = pd.read_csv("translated_partial_ru.csv")
text_raw_ru = " ".join(df["dialog_ru"].to_list()).lower()
len(text_raw_ru)

In [ ]:
text_clean_ru_list = re.findall(r"[а-яё\-]+", text_raw_ru)
len(set(text_clean_ru_list))

In [ ]:
word_counter_ru = Counter(text_clean_ru_list)
word_counter_ru.most_common(1000)

In [ ]:
df_words_ru = pd.DataFrame(word_counter_ru.most_common(), columns=["word", "count"])
df_words_ru.sort_values(by="count", ascending=False, inplace=True)
df_words_ru

In [ ]:
def substring(s, n):
    if len(s) < n:
        return [s]
    return [s[i:i+n] for i in range(len(s)-n+1)]

print(substring("abcdefg", 3))




In [ ]:
for n in [3, 4, 5]:
    df_words_ru[f"substr{n}"] = df_words_ru["word"].apply(substring, n=n)
df_words_ru.tail(5)


In [ ]:
substr3_ru_counts = Counter(df_words_ru.substr3.sum())
print(len(substr3_ru_counts))
substr3_ru_counts.most_common()

In [ ]:
substr4_ru_counts = Counter(df_words_ru.substr4.sum())
print(len(substr4_ru_counts))
substr4_ru_counts.most_common()

In [ ]:
substr5_ru_counts = Counter(df_words_ru.substr5.sum())
print(len(substr5_ru_counts))
substr5_ru_counts.most_common()

In [ ]:
df_words_ru.query("count>=20").shape
df_words_ru.query("count>=20").word.tolist()

In [ ]:
# def get_counts(str_list, counter):
#     return [(s, counter.get(s)) for s in str_list if counter.get(s) > 1]

# df_words_ru["substr3_count"] = df_words_ru["substr3"].apply(get_counts, counter=substr3_ru_counts)
# df_words_ru["substr4_count"] = df_words_ru["substr4"].apply(get_counts, counter=substr4_ru_counts)
# df_words_ru["substr5_count"] = df_words_ru["substr5"].apply(get_counts, counter=substr5_ru_counts)
df_words_ru.tail(30)

In [ ]:
df_words_ru[df_words_ru.word.str.contains("отсут")].word.to_list()

In [151]:
from itertools import chain


class TrieNode:
    def __init__(self):
        self.children = {}
        self.count = 0
        self.word_indices = []  # store indices of words passing through


class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word, index):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
            node.count += 1
            node.word_indices.append(index)


def find_longest_prefix_groups(words, min_group_size=2, min_prefix_len=3):
    """
    Iteratively find LONGEST prefixes having >= min_group_size words.
    Optimized: word indices are stored in trie nodes, so we avoid rescanning.
    """
    remaining = list(range(len(words)))  # track indices instead of words
    groups = {}

    while True:
        # Build trie for current remaining indices
        trie = Trie()
        for idx in remaining:
            trie.insert(words[idx], idx)

        new_groups = {}

        def dfs(node, prefix):
            found_child_group = False
            for ch, child in node.children.items():
                if child.count >= min_group_size:
                    dfs(child, prefix + ch)
                    found_child_group = True
            if (
                node.count >= min_group_size
                and not found_child_group
                and len(prefix) >= min_prefix_len
            ):
                new_groups[prefix] = [words[i] for i in node.word_indices]

        dfs(trie.root, "")

        if not new_groups:
            break

        groups.update(new_groups)
        grouped = set(chain.from_iterable(new_groups.values()))
        remaining = [i for i in remaining if words[i] not in grouped]

    singletons = [words[i] for i in remaining]
    return groups, singletons


# Example usage
words = df_words_ru.word.to_list()

prefix_groups, prefix_singletons = find_longest_prefix_groups(
    words, min_group_size=2, min_prefix_len=3
)

print(len(prefix_groups))
print(len(prefix_singletons))
for sub, words in prefix_groups.items():
    print(sub, words)

print(prefix_singletons)

15460
1279
являет ['является', 'являетесь']
являю ['являются', 'являюсь']
явление ['явлением', 'явление']
явно ['явно', 'явном']
языка ['языка', 'языках', 'языками']
языково ['языковой', 'языковом', 'языкового']
языковы ['языковые', 'языковым', 'языковых']
язв ['язва', 'язву']
яйца ['яйца', 'яйцами']
яйце ['яйце', 'яйцеклеток']
япони ['японии', 'японию', 'япония']
японски ['японский', 'японских', 'японские']
японско ['японском', 'японское', 'японского', 'японской']
японц ['японцев', 'японцы']
ясно ['ясно', 'ясность', 'ясного', 'ясное']
ясным ['ясным', 'ясными']
ясл ['яслей', 'ясли']
яблок ['яблоки', 'яблоко', 'яблок', 'яблока']
яблочны ['яблочный', 'яблочных', 'яблочным', 'яблочные']
ящик ['ящик', 'ящика', 'ящики', 'ящике', 'ящиков']
ярки ['яркий', 'яркие', 'ярких', 'ярким']
ярког ['яркого', 'яркоглазым']
ярмарк ['ярмарку', 'ярмарке', 'ярмарка', 'ярмарки']
ярлык ['ярлык', 'ярлыках']
ярост ['ярость', 'ярости']
ярдов ['ярдовой', 'ярдовую']
янг ['янг', 'янга']
январ ['января', 'январь', '

In [155]:
from itertools import chain


class TrieNode:
    def __init__(self):
        self.children = {}
        self.count = 0
        self.word_indices = []  # store indices of words passing through


class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word, index):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
            node.count += 1
            node.word_indices.append(index)


def find_longest_suffix_groups(words, min_group_size=2, min_suffix_len=3):
    """
    Iteratively find LONGEST suffixes having >= min_group_size words.
    Optimized: store word indices at nodes in trie of reversed words.
    """
    remaining = list(range(len(words)))  # indices
    groups = {}

    while True:
        trie = Trie()
        for idx in remaining:
            trie.insert(words[idx][::-1], idx)  # insert reversed word

        new_groups = {}

        def dfs(node, suffix_rev):
            found_child_group = False
            for ch, child in node.children.items():
                if child.count >= min_group_size:
                    dfs(child, suffix_rev + ch)
                    found_child_group = True
            if (
                node.count >= min_group_size
                and not found_child_group
                and len(suffix_rev) >= min_suffix_len
            ):
                suffix = suffix_rev[::-1]  # flip back to original direction
                new_groups[suffix] = [words[i] for i in node.word_indices]

        dfs(trie.root, "")

        if not new_groups:
            break

        groups.update(new_groups)
        grouped = set(chain.from_iterable(new_groups.values()))
        remaining = [i for i in remaining if words[i] not in grouped]

    singletons = [words[i] for i in remaining]
    return groups, singletons

# Example usage
words = df_words_ru.word.to_list()

suffix_groups, suffix_singletons = find_longest_suffix_groups(
    words, min_group_size=2, min_suffix_len=3
)


# Print results
print(len(suffix_groups))
print(len(suffix_singletons))

print("Shared suffix groups:")
for suffix, group_words in suffix_groups.items():
    print(f"'{suffix}' -> {group_words}")

print("\nSingletons (no shared suffix):")
print(suffix_singletons)


14831
1756
Shared suffix groups:
'учителя' -> ['учителя', 'поручителя']
'водителя' -> ['водителя', 'руководителя', 'производителя']
'требителя' -> ['истребителя', 'потребителя']
'рителя' -> ['растворителя', 'ободрителя']
'явителя' -> ['предъявителя', 'заявителя']
'представителя' -> ['представителя', 'компании-представителя']
'сителя' -> ['носителя', 'спасителя']
'отеля' -> ['отеля', 'гранд-отеля', 'мотеля']
'гателя' -> ['двигателя', 'сберегателя']
'вателя' -> ['пользователя', 'преподавателя', 'отбеливателя']
'одателя' -> ['работодателя', 'арендодателя']
'получателя' -> ['получателя', 'грузополучателя']
'стиля' -> ['стиля', 'текстиля']
'юля' -> ['июля', 'кастрюля', 'вестибюля', 'пилюля']
'контроля' -> ['контроля', 'круиз-контроля', 'самоконтроля']
'аля' -> ['фестиваля', 'февраля', 'вуаля', 'виталя', 'миндаля']
'еня' -> ['меня', 'женьшеня', 'оленя', 'бюллетеня']
'дня' -> ['сегодня', 'дня', 'полудня', 'полдня']
'арня' -> ['парня', 'пекарня']
'кухня' -> ['кухня', 'суп-кухня']
'овня' -> ['у

In [99]:
all_sigletons = set(prefix_singletons).intersection(set(suffix_singletons))
print(len(all_sigletons))
print(all_sigletons)

7025
{'давления', 'убийцу', 'жирными', 'делая', 'обречены', 'домашняя', 'нет', 'закончился', 'очереди', 'успею', 'семинаре', 'заставим', 'хорошие', 'нью-гемпшир', 'этику', 'достопримечательностей', 'задние', 'взглянув', 'практике', 'компьютерная', '-плеере', 'служить', 'шути', 'октока', 'университет', 'утомить', 'разный', 'украдены', 'согласятся', 'признательна', 'раскрыв', 'проклятье', 'менеджер', 'заработать', 'химику', 'сквер', 'шелке', 'рыб', 'нес', 'конвенция', 'остыть', 'вэ', 'чип', 'много', 'олимпийские', 'поцелуев', 'забавно', 'озере', 'обмениваем', 'митчум', 'угон', 'пищи', 'традиционных', 'требований', 'программного', 'возникает', 'экскурсии', 'электрическая', 'мириам', 'линингу', 'материнства', 'расслабленный', 'ужасный', 'фэй', 'мескерем', 'цвет', 'бёрд', 'консультации', 'оставаясь', 'сигарету', 'дешев', 'лгать', 'написана', 'терминах', 'вам', 'отставали', 'сегодняшнюю', 'технически', 'фумаролов', 'резина', 'персонаж', 'кошкой', 'границы', 'гиде', 'переверну', 'семеро', 'су

In [ ]:
from collections import defaultdict


def find_shared_substrings(words, min_group_size=2, min_length=2):
    """
    Find all substrings that appear in at least `min_group_size` words.
    Only consider substrings of length >= min_length.
    """
    substring_map = defaultdict(set)  # substring -> set of words

    for word in words:
        n = len(word)
        seen = set()  # avoid counting duplicate substrings in the same word
        # Generate all substrings of length >= min_length
        for start in range(n):
            for end in range(start + min_length, n + 1):
                sub = word[start:end]
                if sub not in seen:
                    substring_map[sub].add(word)
                    seen.add(sub)

    # Filter by min_group_size
    result = [
        (sub, list(word_set))
        for sub, word_set in substring_map.items()
        if len(word_set) >= min_group_size
    ]

    # Optionally, sort by substring length (longest first)
    result.sort(key=lambda x: (-len(x[0]), x[0]))
    return result


# Example usage
words = df_words_ru.word.to_list()

shared_subs = find_shared_substrings(words, min_group_size=2, min_length=2)

# Deduplicate shared_subs by group_words, keep longest sub
deduped = {}
for sub, group_words in shared_subs:
    key = tuple(sorted(group_words))  # make group_words hashable
    if key not in deduped or len(sub) > len(deduped[key]):
        deduped[key] = sub

# Rebuild as list of (sub, group_words)
shared_subs_deduped = {sub: list(key) for key, sub in deduped.items()}

print("Before deduplication:", len(shared_subs))
print("After deduplication:", len(shared_subs_deduped))
print("Deduplicated shared substrings:")
for sub, group_words in shared_subs_deduped.items():
    print(f"'{sub}' -> {group_words}")

In [ ]:
middle_groups = {
    sub: group_words
    for sub, group_words in shared_subs_deduped.items()
    if not group_words[0].startswith(sub) and not group_words[0].endswith(sub) and len(sub)>3
}
print(len(shared_subs_deduped))
print(len(middle_groups))
for sub, group_words in middle_groups.items():
    print(f"'{sub}' -> {group_words}")

In [138]:
prefix_groups["озер"]

['озера', 'озеру', 'озере', 'озер']

In [137]:
df_words_ru[df_words_ru.word.str.contains("озер")].word.to_list()

['озеро',
 'озера',
 'озеру',
 'озере',
 'озер',
 'озером',
 'козерог',
 'цельнозерновой',
 'цельнозернового',
 'цельнозерновые']

In [ ]:
import pymorphy3

morph = pymorphy3.MorphAnalyzer()

df_words_ru["root"] = df_words_ru["word"].apply(lambda x: morph.parse(x)[0].normal_form)
print(df_words_ru.root.nunique())
print(df_words_ru.word.nunique())
df_words_ru_agg = (
    df_words_ru.groupby("root").agg({"word": list}).reset_index(drop=False)
)
df_words_ru_agg["nwords"] = df_words_ru_agg.word.apply(len)
df_words_ru_agg = df_words_ru_agg.sort_values("nwords", ascending=False).reset_index(
    drop=True
)
df_words_ru_agg.to_csv("df_words_ru_agg.csv", encoding="utf-8-sig", index=False)
display(df_words_ru_agg)

15879
41297


,root,word,nwords
0,хороший,"[лучше, хорошая, хороший, хорошие, хорошее, хо...",34
1,использовать,"[использовать, используете, использую, использ...",27
2,сделать,"[сделать, сделал, сделаю, сделали, сделаем, сд...",26
3,высокий,"[высокий, высокая, высокие, высокого, высокой,...",25
4,плохой,"[хуже, плохой, плохая, плохого, плохое, плохие...",23
...,...,...,...
15874,наана,[наан],1
15875,набивать,[набивают],1
15876,набивка,[набивкой],1
15877,набить,[набить],1
